# Sleep Learning Engine on Kaggle (persistent ambient + reliable T4 GPU)

This notebook is the Kaggle counterpart of the Colab one.
Use it when:

- You have a **large ambient library** (50+ tracks) you do not
  want to re-upload every session. Upload it once as a Kaggle
  dataset and read from ``/kaggle/input/<dataset-slug>/``.
- You need a **guaranteed T4 GPU** (Colab is best-effort and may
  assign 0/15 GB). Kaggle gives 2x T4 (30 GB VRAM) and 30 h/week
  of GPU time per account.
- You want the **working directory to persist** between sessions
  so the cached TTS segments and procedural ambients do not
  regenerate every time.

**One-time setup** (in your Kaggle account, not in the notebook):

1. Go to `kaggle.com/datasets/new` and upload your 96 ambient
   tracks. Title the dataset `ambient` (or any kebab-case slug).
2. (Optional) Upload your script.txt and background image as a
   second dataset called `sleep_learning_engine-assets`, so the notebook can
   pick them up without manual upload.
3. Open this notebook in Kaggle, set **Accelerator = GPU T4 x2**
   and **Internet = On** in the Settings panel.
4. Click **+ Add data** in the right panel and add your `ambient`
   dataset (and `sleep_learning_engine-assets` if you made one).
5. Run all cells.

**Cost:** free. 30 GPU hours per week, 20 GB of working storage
that persists across sessions.

In [ ]:
# 1. Install sleep_learning_engine from the public repo. We use
# the tarball URL (not a git clone) because Kaggle's pip does
# not always have working git credentials. The tarball is a
# plain HTTPS download.
!pip install -q https://github.com/fernandojjq/sleep-learning-engine/archive/refs/heads/main.tar.gz

# Verify the runtime actually has a GPU. The ffmpeg encoder list
# is not enough: h264_nvenc is present whenever ffmpeg was built
# with --enable-nvenc, regardless of whether a GPU is bound to
# the container. The real test is nvidia-smi.
import subprocess
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.splitlines()[0]
print(ffmpeg_version)

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"], capture_output=True, text=True, timeout=15
)
if smi.returncode == 0 and smi.stdout.strip():
    print("GPU detected:")
    print(smi.stdout)
    print("Note: NVENC does NOT use VRAM. 0.0/15.0 GB USED is normal -",
          "NVENC runs on dedicated T4 silicon, not on the GPU cores.",
          "If VRAM shows >0 GB, the encode is going through the GPU",
          "and is therefore hardware-accelerated.")
else:
    print("No GPU detected. Encode will fall back to libx264 CPU; 5-10x slower.")
    print("Fix: Settings -> Accelerator -> GPU T4 x2 -> Save.")

nvenc = subprocess.run(
    ["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True
).stdout
if "h264_nvenc" in nvenc:
    print("NVENC encoder: AVAILABLE")
else:
    print("NVENC encoder: NOT in ffmpeg build (libx264 fallback)")

# Replace the system ffmpeg with a modern static build. Both
# Colab and Kaggle ship ffmpeg 4.4.x which has known issues
# with the audio filter graph the studio uses (sidechaincompress
# + alimiter + concat demuxer - the filter graph parser bails
# out with 'Stream specifier matches no streams'). The static
# build is NVENC-capable, has every audio filter we need, and is
# what the rest of the notebook will find on PATH after this cell
# runs. ~80 MB download, ~30 s on a typical Colab / Kaggle
# connection.
import subprocess
ffmpeg_version = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout
if "ffmpeg version 4." in ffmpeg_version:
    print("Upgrading ffmpeg (system is 4.4.x, too old for the studio filter graph)...")
    ffmpeg_install_cmd = (
        "set -e; "
        "curl -sL https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz "
        "| tar xJ; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/ffmpeg; "
        "cp ffmpeg-master-latest-linux64-gpl/bin/ffprobe /usr/local/bin/ffprobe; "
        "chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe"
    )
    subprocess.check_call(['bash', '-c', ffmpeg_install_cmd])
    print("ffmpeg upgraded.")
else:
    print("ffmpeg is already modern enough.")
print(subprocess.run(
    ["ffmpeg", "-version"], capture_output=True, text=True
).stdout.split('\n')[0])

# === CONFIG: edit this dict and re-run to change the render. ===
# The cell writes the values to /kaggle/working/.sleeplens.toml
# and exports SLEEP_LEARNING_ENGINE_HOME so the CLI finds it.
# Default values match the project's baseline (Brian voice,
# Full list of Edge TTS voice ids: https://learn.microsoft.com/
CONFIG = {
    "tts_voice": "en-US-BrianNeural",  # Edge TTS voice id
    "tts_rate": "-5%",                   # Edge TTS rate (e.g. -10% slower)
    "tts_pitch": "-2Hz",                 # Edge TTS pitch (e.g. -4Hz lower)
    "ambient_volume": 0.34,             # 0.0 (silent) to 1.0 (max)
    "ambient_duck_db": 12.0,            # dB cut on the bed while voice speaks
    "pause_between_paragraphs": 0.0,     # 0 = none; 1.8 was the old buggy default
    "language": "en",                    # script + voice language
    "output_preset": "sleep_720p",       # sleep_720p or sleep_1080p or audio_only
    "hardware_accel": "auto",           # auto | nvenc | qsv | amf | libx264
    "render_threads": 1,                # 0 = auto, 1 = safest on shared Colab
}

import os
WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
config_path = f"{WORK}/.sleeplens.toml"
toml_lines = [
    f"# Generated by the notebook - edit the CONFIG dict above and re-run this cell.",
]
for k, v in CONFIG.items():
    if isinstance(v, str):
        toml_lines.append(f'{k} = "{v}"')
    elif isinstance(v, bool):
        toml_lines.append(f"{k} = {str(v).lower()}")
    else:
        toml_lines.append(f"{k} = {v}")
with open(config_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(toml_lines) + "\n")

print(f"Config written to {config_path}")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")

# Pin the project root to /kaggle/working so load_settings finds the toml.
os.environ["SLEEP_LEARNING_ENGINE_HOME"] = WORK

# VOICE / RATE etc. are also exposed as plain variables for the
# render cell to use in shell commands without re-reading the toml.
VOICE = CONFIG["tts_voice"]

In [ ]:
# 2. Set up the paths. Kaggle's filesystem is split:
#   /kaggle/input/  - read-only, where your datasets live
#   /kaggle/working/ - writable, persists between sessions
# Sleep Learning Engine writes mixed audio, encoded video, and cached TTS
# segments, so we point all of that at /kaggle/working/.
import os, shutil, glob

WORK = "/kaggle/working"
ASSETS = f"{WORK}/assets"
CACHE = f"{WORK}/cache"
OUTPUT = f"{WORK}/output"

for d in (ASSETS, CACHE, OUTPUT, f"{ASSETS}/ambient"):
    os.makedirs(d, exist_ok=True)

# Copy the ambient library from the read-only dataset into the
# working directory so the sleep_learning_engine scanner can read it. This
# is a one-time copy per session; subsequent cells skip files
# that already exist.
AMBIENT_SRC_SLUG = "ambient"  # change to your dataset slug if different
AMBIENT_SRC = f"/kaggle/input/{AMBIENT_SRC_SLUG}"
AMBIENT_DST = f"{ASSETS}/ambient"

if os.path.exists(AMBIENT_SRC):
    copied = 0
    for ext in ("mp3", "ogg", "wav", "flac", "m4a", "aac"):
        for src in glob.glob(f"{AMBIENT_SRC}/*.{ext}"):
            dst = os.path.join(AMBIENT_DST, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                copied += 1
    total = len(glob.glob(f"{AMBIENT_DST}/*"))
    print(f"Ambient library: {total} tracks ({copied} new this session)")
else:
    print(f"WARNING: {AMBIENT_SRC} not found.")
    print("Add your ambient dataset via the + Add data panel.")

In [ ]:
# 3. (Optional) Render the script with a topic. Set
# USE_TOPIC = True to skip the script.txt upload and have Kaggle
# call your AI provider directly. Leave it False to use the
# script.txt that lives in your `sleep_learning_engine-assets` dataset.
import os, subprocess

USE_TOPIC = False  # default: upload script.txt via dataset. Flip to True to generate in-notebook.
TOPIC = ""  # e.g. "the discovery of penicillin"
TARGET_WORDS = 4500
API_KEY = ""  # your NVIDIA NIM key (or OpenAI key, or ...)
BASE_URL = "https://integrate.api.nvidia.com/v1"
MODEL = "deepseek-ai/deepseek-v4-flash"

if USE_TOPIC:
    if not TOPIC or not API_KEY:
        raise SystemExit("Set TOPIC and API_KEY first.")
    from openai import OpenAI
    from sleep_learning_engine.ai.script_writer import SYSTEM_PROMPT
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=180.0)
    user_prompt = f"Topic: {TOPIC}\nTarget word count: {TARGET_WORDS}."
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.7, max_tokens=8192,
    )
    generated = response.choices[0].message.content
    safe_topic = TOPIC.replace(" ", "_")[:40]
    SCRIPT = f"{ASSETS}/{safe_topic}.txt"
    with open(SCRIPT, "w", encoding="utf-8") as fh:
        fh.write(generated)
    print(f"Generated {len(generated.split())} words -> {SCRIPT}")
else:
    # Use the script from the sleep_learning_engine-assets dataset. Adjust the
    # slug to match your dataset name.
    SCRIPT = "/kaggle/input/sleep_learning_engine-assets/prueba.txt"
    BG_IMAGE = "/kaggle/input/sleep_learning_engine-assets/background.jpg"
    BG_VIDEO = "/kaggle/input/sleep_learning_engine-assets/background.mp4"
    print(f"Script: {SCRIPT}")
    print(f"Background image: {BG_IMAGE}")

In [ ]:
# 4. Render. The ffmpeg encode uses real NVENC on the T4, so a
# 6-minute 1080p video finishes in about 1-2 minutes.
import os, subprocess, sys

# On Kaggle (and Colab), ffmpeg spawned as a subprocess from
# Python does NOT inherit the LD_LIBRARY_PATH that points at the
# NVIDIA driver libraries. Without them, the NVENC encoder can
# fail silently or fall back to a software emulation that is 5-10x
# slower. Patch the env before the subprocess.Popen call.
NVIDIA_LIB_PATHS = [
    "/usr/lib64-nvidia",          # Colab default
    "/usr/lib/x86_64-linux-gnu",  # Debian / Ubuntu / Kaggle
    "/usr/local/cuda/lib64",      # Manual CUDA install
]
nvidia_path = next((p for p in NVIDIA_LIB_PATHS if os.path.exists(p)), None)
if nvidia_path:
    print(f"NVIDIA libs: patching LD_LIBRARY_PATH to include {nvidia_path}")
    os.environ["LD_LIBRARY_PATH"] = nvidia_path + ":" + os.environ.get("LD_LIBRARY_PATH", "")

OUT_STEM = "sleep_learning_engine-kaggle"
cmd = [
    "python", "-m", "sleep_learning_engine", "render",
    "--script", SCRIPT,
    "--output-stem", OUT_STEM,
]
if os.path.exists(BG_IMAGE):
    cmd += ["--background-image", BG_IMAGE]
elif os.path.exists(BG_VIDEO):
    cmd += ["--background-video", BG_VIDEO]

# SLEEP_LEARNING_ENGINE_HOME points the pipeline at WORK so the
# resolved ambient_dir (where cell 2 copied the dataset) and
# output_dir (where the MP4 lands) are both correct. Without
# this env var the auto-detect walks up from the pip-installed
# package and lands in /usr/local/lib/python3.12/dist-packages/.../,
# which is not where the user's assets/output are.
#
# SLEEP_LEARNING_ENGINE_TTS_VOICE lets the user swap voice by
# editing the single VOICE line at the top of cell 1.
env = {
    **os.environ,
    "TMP": CACHE, "TEMP": CACHE,
    "SLEEP_LEARNING_ENGINE_HOME": WORK,
    "SLEEP_LEARNING_ENGINE_TTS_VOICE": VOICE,
}
if nvidia_path:
    env["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]
print(f"Project root: {WORK}  |  Voice: {VOICE}")
print("Running:", " ".join(cmd))
print("-" * 60)
print("STREAMING OUTPUT (Ctrl+C to interrupt):")
print("-" * 60)

# Popen (not run) so each progress line is printed as it arrives.
# The previous version used subprocess.run(..., capture_output=True)
# which buffered the entire render and looked frozen.
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    text=True,
    bufsize=1,
    env=env,
    cwd=WORK,
)
try:
    for line in proc.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate()
    raise
proc.wait()
print("-" * 60)
if proc.returncode != 0:
    raise SystemExit(f"Render failed with exit code {proc.returncode}")
print("Render finished.")

In [ ]:
# 5. Locate the final MP4 and print its path. The file lives in
# /kaggle/working/output/ and survives this session, so you can
# come back tomorrow, open the same notebook, and the file is
# still there to download.
import glob, os

candidates = sorted(glob.glob(f'/kaggle/working/output/{OUT_STEM}.mp4'))
if not candidates:
    raise SystemExit("No MP4 was produced. Check the render cell for errors.")
output = candidates[0]
print(f"Final video: {output}")
print(f"Size: {os.path.getsize(output)/1e6:.1f} MB")
print("")
print("To download:")
print("  - In the Kaggle UI, expand the right panel Output")
print("  - Click the file -> Download")
print("  - Or copy it to a dataset with:")
print(f"    !cp {output} /kaggle/input/<your-dataset>/")

## When to use this notebook vs the Colab one

- **Use Kaggle** (this notebook) when you have a large ambient
  library, want a guaranteed T4, and run renders regularly.
  30 GPU hours per week is more than enough for a backlog of
  scripts.
- **Use Colab** (`docs/cloud/low_ram_render.ipynb`) for a quick
  one-off render with no setup. Internet, GPU, and files are
  ready in 30 seconds. The trade-off is that the GPU is
  best-effort and may not be available at peak times.
- **Use the local GUI** (`uv run python run.py`) when you have
  8+ GB of free RAM and want full offline control.

## Troubleshooting

- **`/kaggle/input/ambient` not found.** You forgot to add the
  dataset via the + Add data panel. Add it, then re-run cell 2.
- **Render is slow (10+ min).** Your runtime is on CPU. Check
  Settings -> Accelerator and pick GPU T4 x2. Kaggle usually
  grants the T4 within seconds; if it does not, save the version
  and try again in a few minutes.
- **Edge TTS rate-limited.** Microsoft throttles to about 1
  request per second on the free service. For a 35-segment
  script, plan for ~40 s of TTS time on top of the encode.
- **Out of /kaggle/working space.** 20 GB cap. If your script
  is very long, the cached TTS segments can fill it. Run
  `!rm -rf /kaggle/working/cache/*` between renders.